# Claude **SKILL.md** — demo (Financial Services)

**Industry chosen:** **Financial Services.** We build an `earnings-summary` skill that
turns raw quarterly numbers into a standardized internal *investor-update memo*.

The notebook has two parts:

| Part | What it shows | Needs |
|------|---------------|-------|
| **A — Skills engine (from scratch)** | We re-implement *progressive disclosure* on top of the normal Messages API, so you can literally watch Claude load the skill in stages. | An Anthropic API key |
| **B — Production path** | The official managed **Skills API** (`container` + `skill_id` + code-execution), the way you'd ship it. | API key + the Skills betas enabled |


## The industry & use case: **Financial Services**

Financial Services is one of Anthropic's named solution verticals, and it's a great fit
for Skills because the work is **repetitive, format-heavy, and must be exact**:

- Every analyst should produce the **same memo structure** and the **same ratios**.
- The arithmetic (margins, growth rates) must be **deterministic** — a job for *code*,
  not for the model guessing.
- A **mandatory compliance disclaimer** must appear every single time.

Our `earnings-summary` skill encodes all three: house format (instructions), exact math
(a bundled Python script), and the disclaimer (a reference file). It only *summarizes
numbers the user provides* — it never gives buy/sell advice.


## Step 1 — Installation

In [1]:
!pip -q install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 15.4 MB/s eta 0:00:00


In [3]:
import os, getpass, anthropic

key = None
# Try Colab Secrets first
try:
    from google.colab import userdata
    key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass
# Fall back to a hidden prompt
if not key:
    key = getpass.getpass("Enter your ANTHROPIC_API_KEY: ")

client = anthropic.Anthropic(api_key=key)

# Current model string at time of writing — change if you like.
MODEL = "claude-sonnet-4-6"
print("Client ready. Using model:", MODEL)

Client ready. Using model: claude-sonnet-4-6


## Step 2 — Build the Skill on disk

A skill is *just files*, so we create the folder and write the three files. Each
`%%writefile` cell below writes one real file you could copy straight into Claude Code
or zip up for the API.

First, create the directory tree:

In [4]:
import os
SKILLS_ROOT = "skills"
SKILL_DIR   = os.path.join(SKILLS_ROOT, "earnings-summary")
os.makedirs(os.path.join(SKILL_DIR, "scripts"), exist_ok=True)
os.makedirs(os.path.join(SKILL_DIR, "references"), exist_ok=True)
print("Created", SKILL_DIR)

Created skills/earnings-summary


### 2a. `SKILL.md` — metadata + instructions

Note how the **`description`** says *what* (turn quarterly figures into a memo) **and
when** (whenever the user supplies quarterly numbers). That sentence is the trigger.
The body tells Claude the exact workflow: **run the script → read the style guide →
write → append the disclaimer.**

In [5]:
%%writefile skills/earnings-summary/SKILL.md
---
name: earnings-summary
description: Turn raw quarterly financial figures into a standardized internal investor-update memo. Use this whenever the user provides quarterly numbers (revenue, COGS, opex, net income) for one or more periods and asks for an earnings summary, results memo, quarterly review, or investor update. Always compute the ratios with the bundled script and follow the house format and disclaimer rules.
---

# Earnings Summary Memo

This skill produces a consistent **internal investor-update memo** from raw quarterly
figures. It exists so every analyst produces the same structure, the same ratios,
and the same mandatory disclaimer, without re-explaining house style each time.

## When to use

Use this skill when the user supplies quarterly financials and wants a written
summary. Do NOT give buy/sell/hold recommendations or price targets — this skill
only *summarizes provided numbers*.

## Workflow (follow in order)

1. **Compute the metrics deterministically.** Do not do the arithmetic yourself.
   Run the bundled script with the user's figures as JSON on standard input:

       run_skill_script(script="finance_metrics.py", args_json=<the user's numbers as JSON>)

   The script returns margins (gross / operating / net) and growth rates
   (quarter-over-quarter and year-over-year). Using code guarantees the numbers
   are correct and identical every run.

2. **Load the house style.** Read `references/style_guide.md` to get the exact
   section order, tone, rounding rules, and the mandatory disclaimer text.

3. **Write the memo** using ONLY the values returned by the script, following the
   section order from the style guide. Round exactly as the style guide says.

4. **Append the mandatory disclaimer** verbatim from the style guide. Never omit it.

## Expected input shape

    {
      "company": "Acme Corp",
      "quarter": "Q2 FY2026",
      "current":       {"revenue": 0, "cogs": 0, "opex": 0, "net_income": 0},
      "prior_quarter": {"revenue": 0, "net_income": 0},
      "year_ago":      {"revenue": 0, "net_income": 0}
    }

`prior_quarter` and `year_ago` are optional — if a period is missing, the script
omits the related growth figure and the memo simply does not mention it.

Writing skills/earnings-summary/SKILL.md


### 2b. `scripts/finance_metrics.py` — the deterministic calculator

This is **Level 3 code**. In a real skill its *source never enters Claude's context* —
Claude just runs it and reads the printed JSON. That's why code is perfect for math:
cheap, exact, repeatable.

In [8]:
%%writefile skills/earnings-summary/scripts/finance_metrics.py
#!/usr/bin/env python3
"""Deterministic financial-ratio calculator for the earnings-summary skill.
Reads one JSON object from STDIN, prints one JSON object to STDOUT."""
import json, sys

def pct_change(new, old):
    if old in (None, 0) or new is None:
        return None
    return round((new - old) / old * 100, 1)

def margin(part, whole):
    if whole in (None, 0) or part is None:
        return None
    return round(part / whole * 100, 1)

def main():
    data = json.load(sys.stdin)
    cur = data.get("current", {})
    revenue, cogs = cur.get("revenue"), cur.get("cogs")
    opex, net_income = cur.get("opex"), cur.get("net_income")

    gross_profit = revenue - cogs if (revenue is not None and cogs is not None) else None
    operating_income = (gross_profit - opex) if (gross_profit is not None and opex is not None) else None

    result = {
        "company": data.get("company"),
        "quarter": data.get("quarter"),
        "revenue": revenue,
        "gross_profit": gross_profit,
        "operating_income": operating_income,
        "net_income": net_income,
        "gross_margin_pct": margin(gross_profit, revenue),
        "operating_margin_pct": margin(operating_income, revenue),
        "net_margin_pct": margin(net_income, revenue),
        "revenue_qoq_pct": pct_change(revenue, data.get("prior_quarter", {}).get("revenue")),
        "revenue_yoy_pct": pct_change(revenue, data.get("year_ago", {}).get("revenue")),
        "net_income_yoy_pct": pct_change(net_income, data.get("year_ago", {}).get("net_income")),
    }
    json.dump(result, sys.stdout, indent=2)

if __name__ == "__main__":
    main()

Overwriting skills/earnings-summary/scripts/finance_metrics.py


### 2c. `references/style_guide.md` — loaded only when needed

The house style is verbose, so we keep it **out of the metadata** and let Claude read it
on demand. This is progressive disclosure in action: zero token cost until the skill fires.

In [9]:
%%writefile skills/earnings-summary/references/style_guide.md
# House Style Guide — Earnings Summary Memo

## Section order (use these exact headings)
1. `# {Company} — {Quarter} Earnings Summary`
2. `## Headline` — one sentence: revenue, the most striking growth figure, and net result.
3. `## Key Metrics` — a bullet list of the figures returned by the script.
4. `## Read-through` — 2-3 sentences interpreting the numbers (margins improving/compressing,
   growth accelerating/slowing). Stay descriptive; do not speculate about the future.
5. `## Disclaimer` — the mandatory text below, verbatim.

## Tone
* Neutral, factual, concise. Third person. No hype words ("amazing", "incredible").
* Describe what the numbers *show*, never what an investor *should do*.

## Rounding & formatting rules
* Currency in millions, one decimal, with `$`, e.g. `$12.3M` (inputs here are in dollars,
  so divide by 1,000,000).
* Percentages: one decimal, signed for growth, e.g. `+8.4%`, `-2.1%`.
* If a growth figure is null/missing, omit that line — do not write "N/A".

## Mandatory disclaimer (copy verbatim)
> This memo summarizes figures provided by the requester and is for internal
> informational purposes only. It is not investment advice, a recommendation, or
> an offer to buy or sell any security.

Overwriting skills/earnings-summary/references/style_guide.md


Confirm the files exist:

In [10]:
!find skills -type f | sort

skills/earnings-summary/references/style_guide.md
skills/earnings-summary/scripts/finance_metrics.py
skills/earnings-summary/SKILL.md


## Step 3 — Level 1: discover skills & build a *metadata-only* system prompt

This mirrors what Claude does at startup: it reads **only** the frontmatter of every
installed skill and puts those one-liners in its system prompt. The full body stays on
disk. Below we parse the frontmatter and assemble that lightweight prompt.

In [11]:
import re, os

def parse_frontmatter(md_text):
    """Split a SKILL.md into (metadata dict, body string)."""
    m = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)$", md_text, re.DOTALL)
    if not m:
        raise ValueError("SKILL.md is missing YAML frontmatter")
    meta = {}
    for line in m.group(1).splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip()] = v.strip()
    return meta, m.group(2)

def discover_skills(root):
    """Find every <root>/<name>/SKILL.md and read just its metadata (Level 1)."""
    found = {}
    for name in sorted(os.listdir(root)):
        path = os.path.join(root, name, "SKILL.md")
        if os.path.isfile(path):
            meta, _ = parse_frontmatter(open(path).read())
            found[meta["name"]] = {"dir": os.path.join(root, name),
                                   "description": meta["description"]}
    return found

SKILLS = discover_skills(SKILLS_ROOT)

# Build the Level-1 system prompt: names + descriptions ONLY.
catalog = "\n".join(f"- {n}: {s['description']}" for n, s in SKILLS.items())
SYSTEM_PROMPT = f"""You are a financial-operations assistant with access to Skills.

Available skills (metadata only — you must load a skill before using it):
{catalog}

When a user request matches a skill's description:
1. FIRST call read_skill_file(skill, path="SKILL.md") to load its instructions.
2. Then follow those instructions exactly, reading referenced files and running
   bundled scripts via the provided tools as needed.
Do not invent numbers; obtain them from the skill's script."""

print(SYSTEM_PROMPT)

You are a financial-operations assistant with access to Skills.

Available skills (metadata only — you must load a skill before using it):
- earnings-summary: Turn raw quarterly financial figures into a standardized internal investor-update memo. Use this whenever the user provides quarterly numbers (revenue, COGS, opex, net income) for one or more periods and asks for an earnings summary, results memo, quarterly review, or investor update. Always compute the ratios with the bundled script and follow the house format and disclaimer rules.

When a user request matches a skill's description:
1. FIRST call read_skill_file(skill, path="SKILL.md") to load its instructions.
2. Then follow those instructions exactly, reading referenced files and running
   bundled scripts via the provided tools as needed.
Do not invent numbers; obtain them from the skill's script.


## Step 4 — Define the runtime tools (how Claude reaches into the skill)

In the real product, Claude uses **bash** to `cat` skill files and run scripts. We give it
two safe, equivalent tools:

- **`read_skill_file`** → returns the text of a file inside the skill (this is how the
  `SKILL.md` body and the `references/` files reach Claude — **Levels 2 & 3**).
- **`run_skill_script`** → runs a bundled script, feeding it JSON on stdin and returning
  only its **stdout**. The script's *source code never enters the conversation* — exactly
  the efficiency win Skills are designed for.

`safe_join` blocks path-traversal so Claude can only read inside the skill folder.

In [12]:
import json, subprocess

def safe_join(base, rel):
    """Resolve `rel` inside `base`, refusing anything that escapes the folder."""
    full = os.path.normpath(os.path.join(base, rel))
    base_n = os.path.normpath(base)
    if full != base_n and not full.startswith(base_n + os.sep):
        raise ValueError(f"Refusing to read outside the skill: {rel}")
    return full

def read_skill_file(skill, path):
    if skill not in SKILLS:
        return f"ERROR: unknown skill '{skill}'"
    target = safe_join(SKILLS[skill]["dir"], path)
    if not os.path.isfile(target):
        return f"ERROR: file not found: {path}"
    return open(target).read()

def run_skill_script(skill, script, args_json):
    if skill not in SKILLS:
        return f"ERROR: unknown skill '{skill}'"
    script_path = safe_join(SKILLS[skill]["dir"], os.path.join("scripts", script))
    if not os.path.isfile(script_path):
        return f"ERROR: script not found: {script}"
    # args_json may arrive as a dict or a JSON string — normalise to a string.
    if isinstance(args_json, (dict, list)):
        args_json = json.dumps(args_json)
    proc = subprocess.run(["python3", script_path], input=args_json,
                           capture_output=True, text=True, timeout=30)
    return proc.stdout if proc.returncode == 0 else f"ERROR:\n{proc.stderr}"

# Tool schemas advertised to Claude
TOOLS = [
    {
        "name": "read_skill_file",
        "description": "Read a text file inside a skill folder (e.g. 'SKILL.md' or 'references/style_guide.md').",
        "input_schema": {
            "type": "object",
            "properties": {
                "skill": {"type": "string"},
                "path":  {"type": "string", "description": "path relative to the skill folder"},
            },
            "required": ["skill", "path"],
        },
    },
    {
        "name": "run_skill_script",
        "description": "Run a bundled script (in the skill's scripts/ folder), passing args_json on stdin; returns stdout.",
        "input_schema": {
            "type": "object",
            "properties": {
                "skill":     {"type": "string"},
                "script":    {"type": "string"},
                "args_json": {"type": "object", "description": "the input object passed to the script on stdin"},
            },
            "required": ["skill", "script", "args_json"],
        },
    },
]

DISPATCH = {"read_skill_file": read_skill_file, "run_skill_script": run_skill_script}
print("Tools defined:", [t["name"] for t in TOOLS])

Tools defined: ['read_skill_file', 'run_skill_script']


## Step 5 — The agent loop

A tool-using conversation is a loop:

1. Send the messages to Claude.
2. If Claude's `stop_reason` is `"tool_use"`, run the requested tool(s), append the
   results, and loop again.
3. When Claude stops asking for tools, it has produced the final answer.

The `trace=True` printing lets you **watch progressive disclosure happen** — you'll see
Claude load `SKILL.md`, then read the style guide, then run the script, in that order.

In [13]:
def run_agent(user_prompt, trace=True):
    messages = [{"role": "user", "content": user_prompt}]
    while True:
        resp = client.messages.create(
            model=MODEL, max_tokens=2000,
            system=SYSTEM_PROMPT, tools=TOOLS, messages=messages,
        )
        # Show Claude's thinking-out-loud text and any tool calls
        for block in resp.content:
            if block.type == "text" and block.text.strip() and trace:
                print("🟦 Claude:", block.text.strip()[:400])
            if block.type == "tool_use" and trace:
                print(f"🛠️  calls {block.name}({json.dumps(block.input)[:160]})")

        if resp.stop_reason != "tool_use":
            # Final answer: return the concatenated text blocks
            return "".join(b.text for b in resp.content if b.type == "text")

        # Feed every tool result back
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for block in resp.content:
            if block.type == "tool_use":
                out = DISPATCH[block.name](**block.input)
                if trace:
                    print(f"   ↳ result: {out[:200].replace(chr(10),' ')}…\n")
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": out})
        messages.append({"role": "user", "content": results})

print("run_agent() ready")

run_agent() ready


## Step 6 — Run the demo 🎬

We give Claude *only* raw numbers and a plain request. Notice we never tell it the memo
format or the disclaimer — that all comes from the skill. Watch the trace: Claude reads
the metadata, decides the `earnings-summary` skill applies, loads it, runs the script,
reads the style guide, and writes the memo.

In [14]:
sample = {
    "company": "Acme Corp",
    "quarter": "Q2 FY2026",
    "current":       {"revenue": 48_200_000, "cogs": 18_900_000, "opex": 15_600_000, "net_income": 9_100_000},
    "prior_quarter": {"revenue": 44_500_000, "net_income": 7_800_000},
    "year_ago":      {"revenue": 39_100_000, "net_income": 6_200_000},
}

user_prompt = (
    "Here are Acme Corp's Q2 FY2026 numbers (in USD). Please produce our standard "
    "earnings summary.\n\n" + json.dumps(sample, indent=2)
)

print("================  TRACE  ================")
memo = run_agent(user_prompt, trace=True)
print("\n================  FINAL MEMO  ================\n")
print(memo)

================  TRACE  ================
🟦 Claude: I'll start by loading the skill instructions and can fetch the style guide in parallel too. Let me first read the SKILL.md to understand exactly what's needed.
🛠️  calls read_skill_file({"skill": "earnings-summary", "path": "SKILL.md"})
   ↳ result: --- name: earnings-summary description: Turn raw quarterly financial figures into a standardized internal investor-update memo. Use this whenever the user provides quarterly numbers (revenue, COGS, op…

🟦 Claude: Now I'll run the finance metrics script and load the style guide simultaneously!
🛠️  calls run_skill_script({"skill": "earnings-summary", "script": "finance_metrics.py", "args_json": {"company": "Acme Corp", "quarter": "Q2 FY2026", "current": {"revenue": 48200000, "co)
🛠️  calls read_skill_file({"skill": "earnings-summary", "path": "references/style_guide.md"})
   ↳ result: {   "company": "Acme Corp",   "quarter": "Q2 FY2026",   "revenue": 48200000,   "gross_profit": 29300000,   "

**What you should observe in the trace**

1. `read_skill_file(earnings-summary, "SKILL.md")` — Level 2 instructions load.
2. `run_skill_script(... finance_metrics.py ...)` — the exact ratios come back as JSON.
3. `read_skill_file(... "references/style_guide.md")` — Level 3 reference loads.
4. A final memo with the right sections, signed-percentage growth, `$M` rounding, and the
   compliance disclaimer — none of which we put in the prompt. The **skill** supplied it.

Try changing `sample` (drop `year_ago`, change numbers). The memo stays consistent and the
math stays correct, because the *script* does the arithmetic — that's the whole point.

## Part B — The production path: the managed **Skills API**

Part A re-built the mechanism by hand so you could see it. In production you don't have to:
the Claude API runs Skills for you inside a **code-execution container**. You either
reference a **pre-built** skill by `skill_id` (e.g. `pptx`, `xlsx`, `docx`, `pdf`) or
**upload your own** `SKILL.md` folder via the `/v1/skills` endpoints and reference the id
it returns.

Three **beta headers** are required:

- `code-execution-2025-08-25` — Skills run in the code-execution container
- `skills-2025-10-02` — enables Skills
- `files-api-2025-04-14` — for files in/out of the container

The cell below shows the documented invocation shape using a **pre-built** skill (no upload
needed). It's wrapped in `try/except` because the betas must be enabled on your account; if
they're not, you'll get a clear message instead of a crash.

In [15]:
# Documented production invocation. Requires the Skills betas on your account.
try:
    resp = client.beta.messages.create(
        model=MODEL,
        max_tokens=1500,
        betas=["code-execution-2025-08-25", "skills-2025-10-02", "files-api-2025-04-14"],
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        container={
            # Reference one or more skills here. 'pdf' is a pre-built skill (no upload).
            "skills": [{"type": "anthropic", "skill_id": "pdf"}]
        },
        messages=[{"role": "user",
                   "content": "Use the PDF skill to create a one-page PDF titled 'Hello Skills'."}],
    )
    print("Beta Skills call succeeded. Response blocks:")
    for b in resp.content:
        print(" -", b.type)
except Exception as e:
    print("Beta Skills path not available on this account/SDK yet:\n ", repr(e))
    print("\nThat's fine — Part A above is the fully working demo. To enable this path, see")
    print("the official guide: https://platform.claude.com/docs/en/build-with-claude/skills-guide")

Beta Skills call succeeded. Response blocks:
 - text
 - server_tool_use
 - text_editor_code_execution_tool_result
 - text
 - server_tool_use
 - text_editor_code_execution_tool_result
 - server_tool_use
 - bash_code_execution_tool_result
 - text


### Uploading your *own* skill (custom Skills)

To ship the `earnings-summary/` folder we built, you (1) zip it, (2) upload it to the
**Skills API** (`/v1/skills`), and (3) pass the returned `skill_id` in `container.skills`
exactly like the pre-built example above. The zip step is shown here; the exact upload call
and its fields are documented (and occasionally updated), so follow the official quickstart
linked below rather than memorising a signature.

In [16]:
import shutil
zip_path = shutil.make_archive("earnings-summary-skill", "zip",
                               root_dir="skills", base_dir="earnings-summary")
print("Zipped skill ready to upload:", zip_path)
print("Upload + reference it per:",
      "https://platform.claude.com/docs/en/build-with-claude/skills-guide")

Zipped skill ready to upload: /content/earnings-summary-skill.zip
Upload + reference it per: https://platform.claude.com/docs/en/build-with-claude/skills-guide
